Serve solo a vedere se env è apposto si può eliminare

In [1]:
net_file  = "./nets/nostri/cross.net.xml"
route_file = "./nets/nostri/cross_flows.rou.xml"

In [2]:
import subprocess
import sumolib

result = subprocess.run(
    [sumolib.checkBinary("sumo"), "-n", "./nets/nostri/cross.net.xml", "-r", "./nets/nostri/cross_flows.rou.xml"],
    capture_output=True,
    text=True
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

STDOUT: Step #0.00 (0ms ?*RT. ?UPS, vehicles TOT 0 ACT 0 BUF 0)                                   
Step #100.00 (0ms ?*RT. ?UPS, vehicles TOT 49 ACT 19 BUF 0)                               
Step #200.00 (1ms ~= 1000.00*RT, ~25000.00UPS, vehicles TOT 89 ACT 25 BUF 0)              
Step #300.00 (0ms ?*RT. ?UPS, vehicles TOT 124 ACT 33 BUF 0)                              
Step #400.00 (0ms ?*RT. ?UPS, vehicles TOT 159 ACT 29 BUF 11)                             
Step #500.00 (0ms ?*RT. ?UPS, vehicles TOT 206 ACT 42 BUF 15)                             
Step #600.00 (0ms ?*RT. ?UPS, vehicles TOT 233 ACT 33 BUF 28)                             
Step #700.00 (0ms ?*RT. ?UPS, vehicles TOT 281 ACT 40 BUF 19)                             
Step #800.00 (0ms ?*RT. ?UPS, vehicles TOT 320 ACT 38 BUF 31)                             
Step #900.00 (0ms ?*RT. ?UPS, vehicles TOT 375 ACT 50 BUF 28)                             
Step #1000.00 (0ms ?*RT. ?UPS, vehicles TOT 415 ACT 47 BUF 35)                    

In [3]:
from sumo_rl.environment.env import SumoEnvironment


env = SumoEnvironment(
   net_file=str(net_file),
   route_file=str(route_file),
   single_agent=True,
   use_gui=False,
   num_seconds=500,
)
obs, info = env.reset()

In [7]:
ts = env.traffic_signals["J3"]

print("Corsie totali:", len(ts.lanes))
for lane, tipo in ts.lanes_type.items():
    print(f"  {lane:30s} → {tipo}")

vehicle_lanes = [l for l, t in ts.lanes_type.items() if t == "vehicle"]
ped_lanes     = [l for l, t in ts.lanes_type.items() if t == "pedestrian"]
print(f"\nVeicolari: {len(vehicle_lanes)}  |  Pedonali: {len(ped_lanes)}")

Corsie totali: 12
  -E1_1                          → vehicle
  -E1_2                          → vehicle
  -E2_1                          → vehicle
  -E2_2                          → vehicle
  -E3_1                          → vehicle
  -E3_2                          → vehicle
  -E0_1                          → vehicle
  -E0_2                          → vehicle
  :J3_w1_0                       → pedestrian
  :J3_w2_0                       → pedestrian
  :J3_w3_0                       → pedestrian
  :J3_w0_0                       → pedestrian

Veicolari: 8  |  Pedonali: 4


In [8]:
# ── CELLA 2: Verifica classificazione corsie ─────────────────────────────────
print(f"Corsie totali:    {len(ts.lanes)}")
print(f"Out lanes totali: {len(ts.out_lanes)}")
print(f"Fasi verdi:       {ts.num_green_phases}")

for lane, tipo in ts.lanes_type.items():
    print(f"  IN  {lane:35s} → {tipo}")
for lane, tipo in ts.out_lanes_type.items():
    print(f"  OUT {lane:35s} → {tipo}")

veh_in  = [l for l,t in ts.lanes_type.items() if t == "vehicle"]
ped_in  = [l for l,t in ts.lanes_type.items() if t == "pedestrian"]
veh_out = [l for l,t in ts.out_lanes_type.items() if t == "vehicle"]
ped_out = [l for l,t in ts.out_lanes_type.items() if t == "pedestrian"]
print(f"\nIN  veicolari: {len(veh_in)}  pedonali: {len(ped_in)}")
print(f"OUT veicolari: {len(veh_out)}  pedonali: {len(ped_out)}")

Corsie totali:    12
Out lanes totali: 12
Fasi verdi:       13
  IN  -E1_1                               → vehicle
  IN  -E1_2                               → vehicle
  IN  -E2_1                               → vehicle
  IN  -E2_2                               → vehicle
  IN  -E3_1                               → vehicle
  IN  -E3_2                               → vehicle
  IN  -E0_1                               → vehicle
  IN  -E0_2                               → vehicle
  IN  :J3_w1_0                            → pedestrian
  IN  :J3_w2_0                            → pedestrian
  IN  :J3_w3_0                            → pedestrian
  IN  :J3_w0_0                            → pedestrian
  OUT :J3_c2_0                            → pedestrian
  OUT :J3_c1_0                            → pedestrian
  OUT :J3_c0_0                            → pedestrian
  OUT E0_1                                → vehicle
  OUT E3_1                                → vehicle
  OUT E1_2                      

In [9]:
# ── CELLA 3: Verifica metriche dopo qualche step ─────────────────────────────
for _ in range(60):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())

density  = ts.get_lanes_density()
queue    = ts.get_lanes_queue()
waiting  = ts.get_accumulated_waiting_time_per_lane()

print(f"Num valori density:  {len(density)}  atteso: {len(ts.lanes)}")
print(f"Num valori queue:    {len(queue)}   atteso: {len(ts.lanes)}")
print(f"Num valori waiting:  {len(waiting)} atteso: {len(ts.lanes)}")
print(f"\nDettaglio per corsia:")
for i, lane in enumerate(ts.lanes):
    tipo = ts.lanes_type[lane]
    print(f"  [{tipo:10s}] {lane:35s}  d={density[i]:.3f}  q={queue[i]:.3f}  w={waiting[i]:.1f}")

Num valori density:  12  atteso: 12
Num valori queue:    12   atteso: 12
Num valori waiting:  12 atteso: 12

Dettaglio per corsia:
  [vehicle   ] -E1_1                                d=0.061  q=0.061  w=25.0
  [vehicle   ] -E1_2                                d=0.554  q=0.416  w=85.0
  [vehicle   ] -E2_1                                d=0.061  q=0.000  w=0.0
  [vehicle   ] -E2_2                                d=0.000  q=0.000  w=0.0
  [vehicle   ] -E3_1                                d=0.061  q=0.061  w=61.0
  [vehicle   ] -E3_2                                d=0.777  q=0.777  w=786.0
  [vehicle   ] -E0_1                                d=0.000  q=0.000  w=0.0
  [vehicle   ] -E0_2                                d=0.428  q=0.428  w=243.0
  [pedestrian] :J3_w1_0                             d=0.152  q=0.152  w=41.0
  [pedestrian] :J3_w2_0                             d=0.455  q=0.455  w=230.0
  [pedestrian] :J3_w3_0                             d=0.303  q=0.152  w=19.0
  [pedestrian] :J3_w0_

In [10]:
# ── CELLA 4: Verifica metriche aggregate e di sistema ────────────────────────
print(f"Pressure:       {ts.get_pressure()}")
print(f"Avg speed:      {ts.get_average_speed():.3f}")
print(f"Total queued:   {ts.get_total_queued()}")
print(f"Total CO2:      {ts.get_total_co2():.1f}")

print(f"\nMetriche sistema (da info):")
ped_keys = [k for k in info.keys() if "pedestrian" in k or "person" in k]
veh_keys = [k for k in info.keys() if "system" in k and k not in ped_keys]
for k in veh_keys:
    print(f"  {k}: {info[k]}")
for k in ped_keys:
    print(f"  {k}: {info[k]}")

env.close()

Pressure:       -25
Avg speed:      0.329
Total queued:   37
Total CO2:      45257.1

Metriche sistema (da info):
  system_total_running: 35
  system_total_backlogged: 14
  system_total_stopped: 27
  system_total_arrived: 104
  system_total_departed: 139
  system_total_teleported: 0
  system_total_waiting_time: 927.0
  system_mean_waiting_time: 26.485714285714284
  system_mean_speed: 2.232087434813279
  system_total_running_pedestrians: 37
  system_total_stopped_pedestrians: 10
  system_total_arrived_pedestrians: 19
  system_total_departed_pedestrians: 56
  system_total_waiting_time_pedestrians: 521.0
  system_mean_waiting_time_pedestrians: 14.08108108108108
  system_mean_speed_pedestrians: 0.9228547994566693
